# 2 SQL DATABASE

**Project:** Irish Milk Supply: Growth, Seasonality and Forecasting

**Author:** Temuulen Bulgan

### Purpose of this notebook
This notebook loads the clean data from notebook 1 into the SQL Server database `dairy`. The database and its tables were created with the scripts in the `sql/` folder.

### Input
`data/processed/ireland_dairy_monthly.csv` and `data/processed/canada_milk_monthly.csv`

### Output
The three tables in the `dairy` database are filled: `dim_date`, `dim_measure` and `fact_monthly`.

## Setup

In [1]:
import pandas as pd
import pyodbc
from sqlalchemy import create_engine, text

#### Check the ODBC driver

Python needs an ODBC driver to connect to SQL Server. I check which drivers are installed on this computer.

In [2]:
# List the ODBC drivers installed on this computer
pyodbc.drivers()

['SQL Server',
 'Microsoft Access Driver (*.mdb, *.accdb)',
 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)',
 'Microsoft Access Text Driver (*.txt, *.csv)',
 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)',
 'ODBC Driver 17 for SQL Server',
 'ODBC Driver 18 for SQL Server']

#### Connect to the database

I connect to the local SQL Server with Windows Authentication, the same way as in SSMS.

In [3]:
# Connection to the dairy database on this computer
engine = create_engine(
    "mssql+pyodbc://@localhost/dairy"
    "?driver=ODBC+Driver+18+for+SQL+Server"
    "&trusted_connection=yes"
    "&TrustServerCertificate=yes",
    fast_executemany=True,
)

# Test the connection: list the tables in the dairy database
pd.read_sql("SELECT name FROM sys.tables ORDER BY name", engine)

,name
0,dim_date
1,dim_measure
2,fact_monthly


## 1. Load the clean data

In [4]:
# Load the clean tables from notebook 1, with the date as the row label
ireland = pd.read_csv("../data/processed/ireland_dairy_monthly.csv", index_col="date", parse_dates=True)
canada = pd.read_csv("../data/processed/canada_milk_monthly.csv", index_col="date", parse_dates=True)

print(ireland.shape, canada.shape)

(620, 7) (606, 1)


## 2. Build and load  `dim_date`

One row per month. The Irish data covers January 1975 to August 2026, which includes every Canadian month, so I use the Irish dates.

The `quota_period` column labels each month by the EU milk quota regime: quotas started in April 1984 and ended on 31 March 2015.

In [5]:
# One row per month, with labels taken from the date
dates = ireland.index

dim_date = pd.DataFrame({
    "date_key": dates,
    "year": dates.year,
    "month": dates.month,
    "month_name": dates.month_name(),
    "quarter": dates.quarter,
})

# Quota period: quota by default, then pre-quota and post-quota by date
dim_date["quota_period"] = "quota"
dim_date.loc[dim_date["date_key"] < "1984-04-01", "quota_period"] = "pre-quota"
dim_date.loc[dim_date["date_key"] >= "2015-04-01", "quota_period"] = "post-quota"

dim_date.head(3)

,date_key,year,month,month_name,quarter,quota_period
0,1975-01-01,1975,1,January,1,pre-quota
1,1975-02-01,1975,2,February,1,pre-quota
2,1975-03-01,1975,3,March,1,pre-quota


In [6]:
# Check: number of months in each quota period, and the dates where each period starts and ends
dim_date.groupby("quota_period")["date_key"].agg(["count", "min", "max"])

,count,min,max
quota_period,,,
post-quota,137,2015-04-01,2026-08-01
pre-quota,111,1975-01-01,1984-03-01
quota,372,1984-04-01,2015-03-01


`dim_date` has 620 months: 111 before quotas (January 1975 to March 1984), 372 under quotas (April 1984 to March 2015) and 137 after quotas (April 2015 to August 2026).

#### Clear the tables

The tables are emptied before loading, so the notebook can be re-run without duplicate rows. `fact_monthly` is cleared first because its foreign keys point to both dimension tables.

In [7]:
# Delete all rows, fact table first because it references both dimensions
with engine.begin() as conn:
    conn.execute(text("DELETE FROM dbo.fact_monthly;"))
    conn.execute(text("DELETE FROM dbo.dim_measure;"))
    conn.execute(text("DELETE FROM dbo.dim_date;"))

In [8]:
# Count the rows in each table
pd.read_sql(
    """
    SELECT 'fact_monthly' AS table_name, COUNT(*) AS n_rows FROM dbo.fact_monthly
    UNION ALL
    SELECT 'dim_measure', COUNT(*) FROM dbo.dim_measure
    UNION ALL
    SELECT 'dim_date', COUNT(*) FROM dbo.dim_date;
    """,
    engine,
)

,table_name,n_rows
0,fact_monthly,0
1,dim_measure,0
2,dim_date,0


All three tables are empty.

#### Insert `dim_date`

`to_sql` adds the rows to the existing table. `if_exists="append"` keeps the table as it was created in `sql/01_create_tables.sql`, with its data types and primary key.

In [9]:
# Add the rows to the existing dim_date table in SQL Server
dim_date.to_sql("dim_date", engine, schema="dbo", if_exists="append", index=False)

-1

#### Check `dim_date` in SQL Server

In [10]:
# Count the months in each quota period, read from SQL Server
pd.read_sql(
    """
    SELECT quota_period,
           COUNT(*)      AS n_months,
           MIN(date_key) AS first_month,
           MAX(date_key) AS last_month
    FROM dbo.dim_date
    GROUP BY quota_period
    ORDER BY first_month;
    """,
    engine,
)

,quota_period,n_months,first_month,last_month
0,pre-quota,111,1975-01-01,1984-03-01
1,quota,372,1984-04-01,2015-03-01
2,post-quota,137,2015-04-01,2026-08-01


SQL Server holds the same 620 months as the DataFrame, with the same counts and dates for each quota period.

## 3. Build and load `dim_measure`

One row per measure. It describes what each number in `fact_monthly` means: its full name, unit, country and source table.

In [11]:
# One row per measure: code, full name, unit, country and source table
dim_measure = pd.DataFrame(
    [
        ("intake_ml",       "Domestic milk intake by creameries and pasteurisers", "million litres",   "Ireland", "CSO AKM01"),
        ("fat_pct",         "Fat content of milk intake",                          "%",                "Ireland", "CSO AKM01"),
        ("protein_pct",     "Protein content of milk intake",                      "%",                "Ireland", "CSO AKM01"),
        ("liquid_sales_ml", "Milk sales for human consumption",                    "million litres",   "Ireland", "CSO AKM02"),
        ("butter_kt",       "Butter production",                                   "thousand tonnes",  "Ireland", "CSO AKM03"),
        ("cheese_kt",       "Cheese production",                                   "thousand tonnes",  "Ireland", "CSO AKM03"),
        ("smp_kt",          "Skimmed milk powder production",                      "thousand tonnes",  "Ireland", "CSO AKM03"),
        ("canada_total_ml", "Total milk production",                               "million litres",   "Canada",  "Statistics Canada 32-10-0113-01"),
    ],
    columns=["measure_code", "measure_name", "unit", "country", "source"],
)

dim_measure

,measure_code,measure_name,unit,country,source
0,intake_ml,Domestic milk intake by creameries and pasteur...,million litres,Ireland,CSO AKM01
1,fat_pct,Fat content of milk intake,%,Ireland,CSO AKM01
2,protein_pct,Protein content of milk intake,%,Ireland,CSO AKM01
3,liquid_sales_ml,Milk sales for human consumption,million litres,Ireland,CSO AKM02
4,butter_kt,Butter production,thousand tonnes,Ireland,CSO AKM03
5,cheese_kt,Cheese production,thousand tonnes,Ireland,CSO AKM03
6,smp_kt,Skimmed milk powder production,thousand tonnes,Ireland,CSO AKM03
7,canada_total_ml,Total milk production,million litres,Canada,Statistics Canada 32-10-0113-01


In [12]:
# Check that every measure code matches a column in the clean data
data_columns = list(ireland.columns) + list(canada.columns)
sorted(data_columns) == sorted(dim_measure["measure_code"])

True

`dim_measure` has 8 measures: 7 for Ireland and 1 for Canada. The codes match the column names of the clean data.

#### Insert `dim_measure`

In [13]:
# Add the rows to the existing dim_measure table in SQL Server
dim_measure.to_sql("dim_measure", engine, schema="dbo", if_exists="append", index=False)

-1

#### Check `dim_measure` in SQL Server

In [14]:
# Read the measures back from SQL Server
pd.read_sql(
    """
    SELECT measure_code, measure_name, unit, country, source
    FROM dbo.dim_measure
    ORDER BY country DESC, measure_code;
    """,
    engine,
)

,measure_code,measure_name,unit,country,source
0,butter_kt,Butter production,thousand tonnes,Ireland,CSO AKM03
1,cheese_kt,Cheese production,thousand tonnes,Ireland,CSO AKM03
2,fat_pct,Fat content of milk intake,%,Ireland,CSO AKM01
3,intake_ml,Domestic milk intake by creameries and pasteur...,million litres,Ireland,CSO AKM01
4,liquid_sales_ml,Milk sales for human consumption,million litres,Ireland,CSO AKM02
5,protein_pct,Protein content of milk intake,%,Ireland,CSO AKM01
6,smp_kt,Skimmed milk powder production,thousand tonnes,Ireland,CSO AKM03
7,canada_total_ml,Total milk production,million litres,Canada,Statistics Canada 32-10-0113-01


SQL Server holds the 8 measures: 7 for Ireland and 1 for Canada.

## 4. Build and load `fact_monthly`

`fact_monthly` has one row per month and measure. The clean data has one column per measure, so it is reshaped from wide to long format.

In [15]:
# Turn the date index into a normal column, because melt works on columns
ireland_wide = ireland.reset_index()

# Reshape to long format: one row per month and measure
ireland_long = ireland_wide.melt(
    id_vars="date",
    var_name="measure_code",
    value_name="value",
)

print(ireland_wide.shape, ireland_long.shape)

# The same month in both shapes: one wide row becomes seven long rows
display(ireland_wide[ireland_wide["date"] == "1980-01-01"])
display(ireland_long[ireland_long["date"] == "1980-01-01"])

(620, 8) (4340, 3)


,date,intake_ml,fat_pct,protein_pct,liquid_sales_ml,butter_kt,cheese_kt,smp_kt
60,1980-01-01,94.5,3.71,3.15,38.2,1.8,0.0,1.6


,date,measure_code,value
60,1980-01-01,intake_ml,94.50
680,1980-01-01,fat_pct,3.71
1300,1980-01-01,protein_pct,3.15
1920,1980-01-01,liquid_sales_ml,38.20
2540,1980-01-01,butter_kt,1.80
3160,1980-01-01,cheese_kt,0.00
3780,1980-01-01,smp_kt,1.60


One month with 7 measures becomes 7 rows. The Irish data in long format has 4,340 rows: 620 months × 7 measures.

In [16]:
# Reshape Canada to long format, the same way as Ireland
canada_long = canada.reset_index().melt(
    id_vars="date",
    var_name="measure_code",
    value_name="value",
)

# Stack the Irish and Canadian rows into one table
fact_monthly = pd.concat([ireland_long, canada_long], ignore_index=True)

print(canada_long.shape, fact_monthly.shape)

(606, 3) (4946, 3)


#### Add Canada and combine

In [17]:
# Reshape Canada to long format, the same way as Ireland
canada_long = canada.reset_index().melt(
    id_vars="date",
    var_name="measure_code",
    value_name="value",
)

# Stack the Irish and Canadian rows into one table
fact_monthly = pd.concat([ireland_long, canada_long], ignore_index=True)

print(canada_long.shape, fact_monthly.shape)

# The point where the Irish rows end and the Canadian rows start
join = len(ireland_long)
display(fact_monthly.iloc[join - 2 : join + 2])

(606, 3) (4946, 3)


,date,measure_code,value
4338,2026-07-01,smp_kt,18.100
4339,2026-08-01,smp_kt,13.600
4340,1976-01-01,canada_total_ml,503.352
4341,1976-02-01,canada_total_ml,461.721


Canada adds 606 rows, one per month. The combined table has 4,946 rows, with the Irish rows first and the Canadian rows after them.

#### Drop the missing values

Months with no value get no row, because `value` is `NOT NULL` in the SQL table. The values are not filled in or estimated.

In [18]:
# Keep only the rows that have a value
n_before = len(fact_monthly)
fact_monthly = fact_monthly.dropna(subset=["value"])

print(n_before, "rows before,", len(fact_monthly), "rows after,", n_before - len(fact_monthly), "dropped")

# Rows per measure, with the first and last month that has a value
fact_monthly.groupby("measure_code")["date"].agg(["count", "min", "max"])

4946 rows before, 4538 rows after, 408 dropped


,count,min,max
measure_code,,,
butter_kt,560,1980-01-01,2026-08-01
canada_total_ml,606,1976-01-01,2026-06-01
cheese_kt,552,1980-01-01,2025-12-01
fat_pct,560,1980-01-01,2026-08-01
intake_ml,620,1975-01-01,2026-08-01
liquid_sales_ml,560,1980-01-01,2026-08-01
protein_pct,560,1980-01-01,2026-08-01
smp_kt,520,1980-01-01,2026-08-01


#### Insert `fact_monthly`

`date` is renamed to `date_key` to match the SQL table.

In [19]:
# Match the column names of the SQL table
fact_monthly = fact_monthly.rename(columns={"date": "date_key"})

# Add the rows to the existing fact_monthly table in SQL Server
fact_monthly.to_sql("fact_monthly", engine, schema="dbo", if_exists="append", index=False)

-1

#### Check `fact_monthly` in SQL Server

The fact table is joined to `dim_measure`, so each count shows the measure's country and unit.

In [20]:
# Rows per measure in SQL Server, with the country and unit from dim_measure
pd.read_sql(
    """
    SELECT m.country,
           f.measure_code,
           m.unit,
           COUNT(*)        AS n_months,
           MIN(f.date_key) AS first_month,
           MAX(f.date_key) AS last_month
    FROM dbo.fact_monthly AS f
    JOIN dbo.dim_measure  AS m
        ON f.measure_code = m.measure_code
    GROUP BY m.country, f.measure_code, m.unit
    ORDER BY m.country DESC, f.measure_code;
    """,
    engine,
)

,country,measure_code,unit,n_months,first_month,last_month
0,Ireland,butter_kt,thousand tonnes,560,1980-01-01,2026-08-01
1,Ireland,cheese_kt,thousand tonnes,552,1980-01-01,2025-12-01
2,Ireland,fat_pct,%,560,1980-01-01,2026-08-01
3,Ireland,intake_ml,million litres,620,1975-01-01,2026-08-01
4,Ireland,liquid_sales_ml,million litres,560,1980-01-01,2026-08-01
5,Ireland,protein_pct,%,560,1980-01-01,2026-08-01
6,Ireland,smp_kt,thousand tonnes,520,1980-01-01,2026-08-01
7,Canada,canada_total_ml,million litres,606,1976-01-01,2026-06-01


SQL Server holds 4,538 rows in `fact_monthly`, with the same counts and dates per measure as the DataFrame. Every row joins to a measure in `dim_measure`.

## Summary

The three tables in the `dairy` database are loaded: `dim_date` (620 months), `dim_measure` (8 measures) and `fact_monthly` (4,538 values). The counts in SQL Server match the DataFrames. Months with no published value have no row in `fact_monthly`.